In [1]:
import time
import logging
import pandas as pd
from truedata import TD_hist

def fetch_truedata_history(
    username: str,
    password: str,
    ticker_list: list,
    duration: str = '1 Y',
    bar_size: str = 'EOD',
    sleep_time: float = 0.1
) -> tuple[pd.DataFrame, list]:
    """
    Fetches historical data from TrueData for a list of tickers.

    Parameters
    ----------
    username : str
        TrueData username.
    password : str
        TrueData password.
    ticker_list : list
        List of ticker symbols to fetch data for.
    duration : str, optional
        Duration of data (e.g., '1 Y', '25 Y', etc.). Default is '1 Y'.
    bar_size : str, optional
        Bar size for data ('EOD', 'WEEK', etc.). Default is 'EOD'.
    sleep_time : float, optional
        Delay between API calls to avoid throttling. Default is 0.2 seconds.

    Returns
    -------
    final_df : pd.DataFrame
        Combined DataFrame of all tickers' historical data.
    error_list : list
        List of tickers that failed to fetch.
    """
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

    # Initialize connection
    td_hist = TD_hist(username, password)

    df_list = []
    error_list = []

    for ticker in ticker_list:
        try:
            df = td_hist.get_historic_data([ticker], duration=duration, bar_size=bar_size)

            df['Ticker'] = ticker
            df = df.rename(columns={
                'timestamp': 'Date',
                'high': 'High',
                'low': 'Low',
                'close': 'Close',
                'open': 'Open'
            })

            df_list.append(df)
            logging.info(f"Fetched data for {ticker} ({len(df)} rows).")
            time.sleep(sleep_time)

        except Exception as e:
            logging.error(f"Failed to fetch data for {ticker}: {e}")
            error_list.append(ticker)

    final_df = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()
    return final_df, error_list


In [2]:
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import yfinance as yf
from pathlib import Path

import os

def run_momentum_strategy(universe_file: str,
                          start_date: str,
                          end_date: str,
                          top_n: int,
                          output_root: str = "Momentum_Results",
                          preserve_until: str | None = None) -> str:
    """
    Run rolling-window momentum strategy on given stock universe.
    """

    # ==== 1. Load Universe ====
    if universe_file.endswith(".csv"):
        stock_list = pd.read_csv(universe_file)[["Symbol", "ISIN Code"]]
    else:
        stock_list = pd.read_excel(universe_file)[["Symbol", "ISIN Code"]]

    username = os.getenv("TRUEDATA_USERNAME")
    password = os.getenv("TRUEDATA_PASSWORD")
    stock_list["Ticker"] = stock_list["Symbol"] 
    stock_list["Ticker"] = stock_list["Ticker"].astype(str).str.strip().str.upper()

    # Permanently ban specific tickers from the portfolio universe before fetching data
    banned_tickers = {"APARINDS"}
    stock_list = stock_list[~stock_list["Ticker"].isin(banned_tickers)].copy()

    symbol_list = stock_list["Ticker"].tolist()
    universe_name = Path(universe_file).stem
    output_dir = os.path.join(output_root, f"{universe_name}_{top_n}_stocks_results")
    os.makedirs(output_dir, exist_ok=True)

    if preserve_until is None and universe_name == 'Nifty_500_2025_Apr' and top_n == 20:
        preserve_until = "2026-04-30"
        print(f"🔒 Freezing data up to {preserve_until} for {universe_name}_{top_n}_stocks_results")

    # ==== 2. Download Data ====
    total_start = pd.to_datetime(start_date)
    total_end = pd.to_datetime(end_date)

    print(f"\n📥 Downloading price data for {len(symbol_list)} symbols...")
    data, errors = fetch_truedata_history(username, password, symbol_list, duration='10 Y', bar_size='EOD')
    data = data[['Date', 'Close', 'Ticker']]
    data.drop_duplicates(subset=['Date', 'Ticker'], inplace=True)

    data['Ticker'] = data['Ticker'].astype(str).str.strip().str.upper()
    prices = data.pivot(index="Date", columns="Ticker", values="Close")
    prices_all = prices.sort_index()

    # ==== 3. Create Rolling Windows ====
    windows = []
    current_start = total_start
    while True:
        current_end = current_start + relativedelta(months=6)
        if current_end > total_end:
            break
        window_prices = prices_all.loc[(prices_all.index >= current_start) & (prices_all.index < current_end)].copy()
        if not window_prices.empty:
            windows.append((current_start, current_end, window_prices))
        current_start += relativedelta(months=1)

    print(f"📊 Created {len(windows)} rolling windows.")

    # ==== 4. Process Each Window ====
    preserve_dt = pd.to_datetime(preserve_until) if preserve_until else None
    for start, end, prices in windows:
        file_suffix = f"{start.strftime('%Y%m%d')}_{end.strftime('%Y%m%d')}"
        print(f"\n🔍 Processing window: {start.date()} → {end.date()}")

        if preserve_dt is not None and end <= preserve_dt:
            print(f"⏭️ Skipping preserved window ending {end.date()} (preserve_until={preserve_dt.date()}).")
            continue

        prices.dropna(axis=1, how='all', inplace=True)
        if prices.empty:
            print("⚠️ All price data missing. Skipping window.")
            continue

        # Monthly momentum
        monthclose = prices.groupby(prices.index.strftime('%Y-%m')).tail(1)
        monthstart = prices.groupby(prices.index.strftime('%Y-%m')).head(1)
        monthstart.index = monthclose.index
        monchange = (monthclose - monthstart) / monthstart
        MOM = (monchange + 1).product() - 1
        mom = MOM * 100

        # Daily returns
        daily_ret = prices.pct_change(fill_method=None)
        positivechange = (daily_ret[daily_ret > 0].count() / daily_ret.count()) * 100
        negativechange = (daily_ret[daily_ret < 0].count() / daily_ret.count()) * 100

        result = pd.concat([positivechange, negativechange, mom], axis=1, join='inner')
        result.columns = ["Positive", "Negative", "Momentum"]
        result = result.reset_index().rename(columns={'index': 'Ticker'})

        result['Ticker'] = result['Ticker'].astype(str).str.strip().str.upper()
        result = pd.merge(result, stock_list[["Ticker", "ISIN Code"]], on="Ticker", how="left")

        try:
            last_prices = prices.tail(1).iloc[0].to_dict()
            last_prices = {str(k).strip().upper(): v for k, v in last_prices.items()}
        except Exception:
            last_prices = {}
        result['Last_Price'] = result['Ticker'].map(last_prices)

        pre_count = len(result)
        result = result[result['Last_Price'].isna() | (result['Last_Price'] <= 7500)]
        removed_count = pre_count - len(result)
        if removed_count:
            print(f"🔒 Excluded {removed_count} stocks with Last_Price > 7500 from this window")

        df = result.copy()
        df["Rank_Mom"] = df["Momentum"].rank(method='min', ascending=False)
        df['FIP'] = df.apply(lambda row: row['Negative'] - row['Positive'] if row['Momentum'] > 0 else np.nan, axis=1)

        df.dropna(inplace=True)
        df["FIP_rank"] = df["FIP"].rank(method="first", ascending=True)
        df["Combined_Rank"] = df["Rank_Mom"] + df["FIP_rank"]
        if end.strftime('%Y-%m-%d') == '2026-01-01':
            df = df[~(df['Ticker'].isin(['MARUTI', 'PTCIL']))]

        df = df.sort_values(by="Combined_Rank", ascending=True).head(top_n)
        df["Real_Rank"] = range(1, len(df) + 1)
        df["End_Date"] = end.strftime('%Y-%m-%d')

        # Manual override: for May 2026 windows, replace ADANIENSOL -> VEDL (keep backups elsewhere)
        try:
            if getattr(end, 'year', None) == 2026 and getattr(end, 'month', None) == 5 and 'ADANIENSOL' in df['Ticker'].astype(str).str.upper().values:
                print(f"🔁 Replacing ADANIENSOL with VEDL for window ending {end.date()}")
                df.loc[df['Ticker'].astype(str).str.upper() == 'ADANIENSOL', 'Ticker'] = 'VEDL'
                if 'ISIN Code' in df.columns:
                    df.loc[df['Ticker'] == 'VEDL', 'ISIN Code'] = 'INE205A01025'
        except Exception:
            pass

        # Prevent duplicate tickers (e.g., VEDL introduced twice after replacement)
        try:
            dup_count = df['Ticker'].duplicated().sum()
            if dup_count > 0:
                print(f"⚠️ Removing {dup_count} duplicate ticker(s) in this window (keeping first occurrence)")
                df = df.drop_duplicates(subset=['Ticker'], keep='first')
                # re-sort and re-rank after dedupe
                df = df.sort_values(by="Combined_Rank", ascending=True).head(top_n)
                df["Real_Rank"] = range(1, len(df) + 1)
        except Exception:
            pass

        # Save each window (overwrite only for non-preserved windows)
        output_file = os.path.join(output_dir, f"momentum_{file_suffix}.xlsx")
        df.to_excel(output_file, index=False)
        print(f"✅ Saved results to: {output_file}")

    # ==== 5. Master File ====
    print("\n📂 Creating master summary file...")
    master_data = []
    for file in os.listdir(output_dir):
        if file.startswith("momentum_") and file.endswith(".xlsx"):
            df = pd.read_excel(os.path.join(output_dir, file))
            selected_df = df[["End_Date", "ISIN Code", "Ticker", 'Real_Rank']].copy()
            master_data.append(selected_df)

    if master_data:
        master_df = pd.concat(master_data, ignore_index=True)
        master_df['Ticker'] = master_df['Ticker'].astype(str).str.strip().str.upper()

        # Enforce May-2026 manual replacement in master: remove/replace ADANIENSOL rows
        try:
            end_dates = pd.to_datetime(master_df['End_Date'], errors='coerce')
            replace_mask = (end_dates.dt.year == 2026) & (end_dates.dt.month == 5) & (master_df['Ticker'] == 'ADANIENSOL')
            if replace_mask.any():
                print(f"🔁 Replacing {replace_mask.sum()} ADANIENSOL rows with VEDL in master (May 2026)")
                master_df.loc[replace_mask, 'Ticker'] = 'VEDL'
                if 'ISIN Code' in master_df.columns:
                    master_df.loc[replace_mask, 'ISIN Code'] = 'INE205A01025'
        except Exception:
            pass

        # Remove duplicate End_Date+Ticker rows in master (prevents VEDL appearing twice for same window)
        try:
            dup_master = master_df.duplicated(subset=['End_Date', 'Ticker'], keep='first').sum()
            if dup_master:
                print(f"⚠️ Removing {dup_master} duplicate End_Date+Ticker rows in master (keeping first)")
                master_df.drop_duplicates(subset=['End_Date', 'Ticker'], keep='first', inplace=True)
        except Exception:
            pass

        master_file_path = os.path.join(output_dir, "master_momentum_summary.xlsx")
        if os.path.exists(master_file_path):
            existing_master = pd.read_excel(master_file_path)
            existing_master['Ticker'] = existing_master['Ticker'].astype(str).str.strip().str.upper()
            combined = pd.concat([existing_master, master_df], ignore_index=True)
            combined.drop_duplicates(subset=['End_Date', 'ISIN Code', 'Ticker'], keep='first', inplace=True)
            combined.to_excel(master_file_path, index=False)
            print(f"✅ Master file updated (appended new rows) at: {master_file_path}")
            return master_file_path
        else:
            master_df.to_excel(master_file_path, index=False)
            print(f"✅ Master file saved to: {master_file_path}")
            return master_file_path
    else:
        print("⚠️ No window files found, master not created.")
        return None

In [3]:



# Momentum/Automating Momentum/Universe/Nifty_500_2025_Apr.csv
# "C:\Users\Admin\Momentum\Automating Momentum\Universe\Nifty_500_2025_Apr.csv"
master_file = run_momentum_strategy(
    universe_file=r"C:\Users\anike\Desktop\Ocean_dev\Momentum Handover\Momentum Handover\new_monthly\ticker_master_may26.xlsx",
    start_date="2022-06-01",
    end_date="2026-05-31",
    top_n=20,
    output_root="Stocks",
    preserve_until="2026-04-30",
)
print("Master summary located at:", master_file)



📥 Downloading price data for 498 symbols...


(2026-05-04 14:38:55,554) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:13808 Thread:12268)
2026-05-04 14:38:55,554 - WARNING - Connected successfully to TrueData Historical Data Service... 
2026-05-04 14:38:55,995 - INFO - Fetched data for 360ONE (1642 rows).
2026-05-04 14:38:56,579 - INFO - Fetched data for 3MINDIA (2476 rows).
2026-05-04 14:38:57,146 - INFO - Fetched data for ABB (2476 rows).
2026-05-04 14:38:57,694 - INFO - Fetched data for ACC (2476 rows).
2026-05-04 14:38:58,210 - INFO - Fetched data for ACMESOLAR (362 rows).
2026-05-04 14:38:58,766 - INFO - Fetched data for AIAENG (2476 rows).
2026-05-04 14:38:59,313 - INFO - Fetched data for APLAPOLLO (2476 rows).
2026-05-04 14:38:59,880 - INFO - Fetched data for AUBANK (2184 rows).
2026-05-04 14:39:00,395 - INFO - Fetched data for AWL (1048 rows).
2026-05-04 14:39:00,888 - INFO - Fetched data for AADHARHFC (487 rows).
2026-05-04 14:39:01,439 - INFO - Fetched data for AARTIIND (2476 rows).
2026-

📊 Created 42 rolling windows.

🔍 Processing window: 2022-06-01 → 2022-12-01
⏭️ Skipping preserved window ending 2022-12-01 (preserve_until=2026-04-30).

🔍 Processing window: 2022-07-01 → 2023-01-01
⏭️ Skipping preserved window ending 2023-01-01 (preserve_until=2026-04-30).

🔍 Processing window: 2022-08-01 → 2023-02-01
⏭️ Skipping preserved window ending 2023-02-01 (preserve_until=2026-04-30).

🔍 Processing window: 2022-09-01 → 2023-03-01
⏭️ Skipping preserved window ending 2023-03-01 (preserve_until=2026-04-30).

🔍 Processing window: 2022-10-01 → 2023-04-01
⏭️ Skipping preserved window ending 2023-04-01 (preserve_until=2026-04-30).

🔍 Processing window: 2022-11-01 → 2023-05-01
⏭️ Skipping preserved window ending 2023-05-01 (preserve_until=2026-04-30).

🔍 Processing window: 2022-12-01 → 2023-06-01
⏭️ Skipping preserved window ending 2023-06-01 (preserve_until=2026-04-30).

🔍 Processing window: 2023-01-01 → 2023-07-01
⏭️ Skipping preserved window ending 2023-07-01 (preserve_until=2026-04

In [4]:
import pandas as pd
import shutil
from pathlib import Path

master_path = Path(r"C:\Users\anike\Desktop\Ocean_dev\Momentum Handover\Momentum Handover\Stocks\Nifty_500_2025_Apr_20_stocks_results\master_momentum_summary.xlsx")
if not master_path.exists():
    raise FileNotFoundError(master_path)

# backup
backup_path = master_path.with_name(master_path.stem + "_backup" + master_path.suffix)
shutil.copy2(master_path, backup_path)

# load, ensure dates
df = pd.read_excel(master_path)
df["End_Date"] = pd.to_datetime(df["End_Date"], errors="coerce")

# mask for May 2026 and ticker match
mask = df["End_Date"].dt.year.eq(2026) & df["End_Date"].dt.month.eq(5) & df["Ticker"].eq("ADANIENSOL")

if mask.any():
    df.loc[mask, "Ticker"] = "VEDL"
    df.to_excel(master_path, index=False)
    print(f"Replaced {mask.sum()} rows. Backup saved to: {backup_path}")
else:
    print("No ADANIENSOL rows found for May 2026. No changes made.")

No ADANIENSOL rows found for May 2026. No changes made.


In [5]:
# ----------------xxxxxxxxxxxxxxx---------------------------------------